# Experience / Seniority Structure Discovery — BGE-base v2

**Goal:** Determine whether the new BGE-base-en-v1.5 embedding space (768-dim) encodes seniority structure more clearly than the previous BGE-small-en-v1.5 (384-dim) model.

**Key change from v1:** Embedding text now includes explicit seniority context (from `job_text.py`):
```
Job Title: <title>
Required Skills: <skills>
Seniority: <level>, <N>+ years experience
Description: <snippet>
```
**v1 baseline (BGE-small, 384-dim):** Mean cluster purity = 0.164 — no seniority structure found.

**Questions re-answered:**
1. Does the embedding space naturally separate by experience level? (UMAP)
2. Do labeled jobs cluster together by position_level? (purity analysis)
3. What k for K-Means produces the best silhouette? (sweep k=8..25)
4. Can a KNN classifier propagate tier labels with reasonable accuracy?

In [ ]:
# Cell 0-A: Install packages (run once, then restart kernel)
import subprocess, sys
pkgs = ["umap-learn", "hdbscan", "matplotlib", "seaborn", "pandas", "pyarrow", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)
print("Packages installed. Restart the kernel now before running remaining cells.")

In [ ]:
# Cell 0-B: Imports and config
import os, json, warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
import psycopg2.extras
from dotenv import load_dotenv

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from scipy.spatial import ConvexHull
import umap
import hdbscan

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.dpi"] = 120

load_dotenv()

OUTPUT_DIR = Path("../data/analysis_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "BAAI/bge-base-en-v1.5"  # 768-dim
EMBEDDING_DIMS = 768

print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Model: {MODEL_NAME} ({EMBEDDING_DIMS}-dim)")

## Section 1 — Data Loading

In [ ]:
# Cell 1-A: Fetch all active jobs with embeddings from Supabase
# job_embeddings.embedding is now vector(768); cast to text for parsing.

QUERY = """
SELECT
    j.job_uuid,
    j.title,
    j.salary_min,
    j.salary_max,
    j.min_years_experience,
    COALESCE(
        NULLIF(TRIM(BOTH '"' FROM (j.position_levels_json::jsonb->0)::text), ''),
        'Unknown'
    ) AS position_level,
    COALESCE(
        NULLIF(TRIM(BOTH '"' FROM (j.categories_json::jsonb->0)::text), ''),
        'Unknown'
    ) AS category,
    e.embedding::text AS embedding_text
FROM jobs j
JOIN job_embeddings e ON e.job_uuid = j.job_uuid
WHERE j.is_active = TRUE
  AND e.embedding IS NOT NULL
"""

DATABASE_URL = os.environ["DATABASE_URL"]
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

print("Fetching jobs... (may take 3-5 min)")
cur.execute(QUERY)
rows = cur.fetchall()
cur.close()
conn.close()

print(f"Fetched {len(rows):,} rows")

In [ ]:
# Cell 1-B: Parse into numpy matrix + DataFrame
# pgvector ::text output is "[f1,f2,...]" — parseable with json.loads.

records = []
embeddings = []

for r in rows:
    try:
        raw = r["embedding_text"]
        if isinstance(raw, str):
            emb = json.loads(raw)
        else:
            emb = list(raw)
        if len(emb) != EMBEDDING_DIMS:
            continue  # skip stale 384-dim rows if any exist
    except (json.JSONDecodeError, TypeError):
        continue

    embeddings.append(emb)
    records.append({
        "job_uuid":            r["job_uuid"],
        "title":               r["title"],
        "salary_min":          r["salary_min"],
        "salary_max":          r["salary_max"],
        "min_years_experience": r["min_years_experience"],
        "position_level":      r["position_level"],
        "category":            r["category"],
    })

X = np.array(embeddings, dtype=np.float32)
df = pd.DataFrame(records)

print(f"Matrix shape: {X.shape}")
print(f"Embedding dim check: {X.shape[1]} (expected {EMBEDDING_DIMS})")
print()
print("Position level distribution:")
print(df["position_level"].value_counts().to_string())
print()
print(f"Jobs with known position_level: {(df['position_level'] != 'Unknown').sum():,} "
      f"({(df['position_level'] != 'Unknown').mean()*100:.1f}%)")
print(f"min_years_experience non-null: {df['min_years_experience'].notna().sum():,}")

## Section 2 — UMAP Dimensionality Reduction

Fit on a 15k sample, transform the full dataset.
Using `metric='cosine'` since BGE embeddings are L2-normalised.

In [ ]:
# Cell 2-A: Fit UMAP
# Expected time: ~10-20 min on CPU (768-dim takes longer than 384-dim)

UMAP_SAMPLE = 15_000
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X), size=min(UMAP_SAMPLE, len(X)), replace=False)
X_sample = X[sample_idx]

print(f"Fitting UMAP on {len(sample_idx):,}-job sample (768-dim)...")
print("(First run compiles numba JIT — expect 2-3 min overhead)")

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.05,
    metric="cosine",
    random_state=42,
    low_memory=False,
)
reducer.fit(X_sample)

print("Transforming full dataset...")
xy = reducer.transform(X)
df["umap_x"] = xy[:, 0]
df["umap_y"] = xy[:, 1]

print(f"Done. UMAP range x={xy[:,0].min():.2f}..{xy[:,0].max():.2f}, "
      f"y={xy[:,1].min():.2f}..{xy[:,1].max():.2f}")

In [ ]:
# Cell 2-B: UMAP visualisations — 3 subplots

PL_PALETTE = {
    "Fresh/entry level":  "#2196F3",
    "Non-executive":      "#64B5F6",
    "Junior Executive":   "#4CAF50",
    "Executive":          "#8BC34A",
    "Professional":       "#9C27B0",
    "Senior Executive":   "#FF9800",
    "Manager":            "#F44336",
    "Middle Management":  "#E91E63",
    "Senior Management":  "#B71C1C",
    "C-Suite/VP":         "#000000",
    "Unknown":            "#EEEEEE",
}

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle(f"UMAP of {len(df):,} jobs — {MODEL_NAME} ({EMBEDDING_DIMS}-dim)", fontsize=13)

# Plot 1: position_level — gray background of unknowns, colored labeled points
ax = axes[0]
ax.scatter(
    df.loc[df["position_level"] == "Unknown", "umap_x"],
    df.loc[df["position_level"] == "Unknown", "umap_y"],
    c="#EEEEEE", s=1, alpha=0.1, linewidths=0, rasterized=True
)
for pl, color in PL_PALETTE.items():
    if pl == "Unknown":
        continue
    sub = df[df["position_level"] == pl]
    if len(sub) == 0:
        continue
    ax.scatter(sub["umap_x"], sub["umap_y"],
               c=color, s=2, alpha=0.5, linewidths=0,
               label=f"{pl} (n={len(sub):,})", rasterized=True)
ax.legend(fontsize=7, loc="lower right", markerscale=3)
ax.set_title("Coloured by position_level", fontsize=11)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")

# Plot 2: min_years_experience (continuous)
ax = axes[1]
has_exp = df["min_years_experience"].notna()
ax.scatter(
    df.loc[~has_exp, "umap_x"], df.loc[~has_exp, "umap_y"],
    c="#EEEEEE", s=1, alpha=0.1, linewidths=0, rasterized=True
)
sc = ax.scatter(
    df.loc[has_exp, "umap_x"], df.loc[has_exp, "umap_y"],
    c=df.loc[has_exp, "min_years_experience"],
    cmap="plasma", s=2, alpha=0.5, vmin=0, vmax=10, linewidths=0, rasterized=True
)
plt.colorbar(sc, ax=ax, label="min_years_experience")
ax.set_title("Coloured by years experience", fontsize=11)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")

# Plot 3: salary_min
ax = axes[2]
has_sal = df["salary_min"].notna() & (df["salary_min"] > 0)
ax.scatter(
    df.loc[~has_sal, "umap_x"], df.loc[~has_sal, "umap_y"],
    c="#EEEEEE", s=1, alpha=0.1, linewidths=0, rasterized=True
)
sc2 = ax.scatter(
    df.loc[has_sal, "umap_x"], df.loc[has_sal, "umap_y"],
    c=np.log1p(df.loc[has_sal, "salary_min"]),
    cmap="viridis", s=2, alpha=0.5, linewidths=0, rasterized=True
)
plt.colorbar(sc2, ax=ax, label="log(salary_min+1)")
ax.set_title("Coloured by salary_min", fontsize=11)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")

plt.tight_layout()
out = OUTPUT_DIR / "umap_overview.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

## Section 3 — K-Means Silhouette Sweep

Sweep k=8..25 to find the optimal cluster count. Uses a 10k subsample for speed.

In [ ]:
# Cell 3-A: Silhouette sweep k=8..25
# Uses a 10k subsample for silhouette scoring — same approach as v1.

SIL_SAMPLE = 10_000
K_RANGE = range(8, 26)

sil_idx = rng.choice(len(X), size=min(SIL_SAMPLE, len(X)), replace=False)
X_sil = X[sil_idx]

silhouette_scores = {}
for k in K_RANGE:
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=5, batch_size=4096)
    km.fit(X)
    labels_sil = km.labels_[sil_idx]
    score = silhouette_score(X_sil, labels_sil, metric="cosine", sample_size=None)
    silhouette_scores[k] = score
    print(f"  k={k:2d}  silhouette={score:.4f}")

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nBest k = {best_k}  (silhouette = {silhouette_scores[best_k]:.4f})")
print("(v1 baseline: best_k=20, silhouette≈0.054)")

In [ ]:
# Cell 3-B: Plot silhouette scores

fig, ax = plt.subplots(figsize=(10, 4))
ks = list(silhouette_scores.keys())
scores = list(silhouette_scores.values())
ax.plot(ks, scores, marker="o", linewidth=1.5)
ax.axvline(best_k, color="red", linestyle="--", label=f"Best k={best_k}")
ax.set_xlabel("k (number of clusters)")
ax.set_ylabel("Silhouette score (cosine)")
ax.set_title(f"K-Means silhouette sweep — {MODEL_NAME} ({EMBEDDING_DIMS}-dim)")
ax.legend()
plt.tight_layout()
out = OUTPUT_DIR / "silhouette_sweep.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

## Section 4 — Final K-Means + Cluster Profiles

In [ ]:
# Cell 4-A: Fit final K-Means with best_k and build cluster profiles

print(f"Fitting final K-Means with k={best_k}...")
km = MiniBatchKMeans(n_clusters=best_k, random_state=42, n_init=10, batch_size=4096)
km.fit(X)
df["kmeans_cluster"] = km.labels_

# Build cluster profiles
cluster_profiles = []
for c in range(best_k):
    sub = df[df["kmeans_cluster"] == c]
    top_titles = sub["title"].value_counts().head(5).to_dict()
    top_categories = sub["category"].value_counts().head(5).to_dict()
    top_levels = sub["position_level"].value_counts().head(5).to_dict()
    profile = {
        "cluster": c,
        "n": len(sub),
        "salary_min_median": float(sub["salary_min"].median()) if sub["salary_min"].notna().any() else None,
        "salary_max_median": float(sub["salary_max"].median()) if sub["salary_max"].notna().any() else None,
        "years_exp_median":  float(sub["min_years_experience"].median()) if sub["min_years_experience"].notna().any() else None,
        "top_titles":     top_titles,
        "top_categories": top_categories,
        "top_levels":     top_levels,
    }
    cluster_profiles.append(profile)

# Sort by median salary (seniority proxy)
cluster_profiles.sort(key=lambda p: p["salary_min_median"] or 0)

print(f"\n{'C':>3}  {'n':>6}  {'SalMin':>8}  {'SalMax':>8}  {'YrsExp':>6}  Top level")
print("-" * 70)
for p in cluster_profiles:
    top_lvl = list(p["top_levels"].keys())[0] if p["top_levels"] else "—"
    print(f"{p['cluster']:>3}  {p['n']:>6,}  "
          f"{(p['salary_min_median'] or 0):>8,.0f}  "
          f"{(p['salary_max_median'] or 0):>8,.0f}  "
          f"{(p['years_exp_median'] or 0):>6.1f}  {top_lvl}")

In [ ]:
# Cell 4-B: UMAP coloured by K-Means cluster

fig, ax = plt.subplots(figsize=(12, 9))
scatter = ax.scatter(
    df["umap_x"], df["umap_y"],
    c=df["kmeans_cluster"], cmap="tab20",
    s=1, alpha=0.3, linewidths=0, rasterized=True
)
plt.colorbar(scatter, ax=ax, label="K-Means cluster")
ax.set_title(f"UMAP coloured by K-Means cluster (k={best_k})", fontsize=12)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
out = OUTPUT_DIR / "umap_kmeans.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

## Section 5 — Cluster Purity Analysis

Key metric: do jobs with the same position_level land in the same clusters?

**v1 baseline:** Mean purity = 0.164 (random baseline ≈ 1/20 = 0.05 for k=20).

In [ ]:
# Cell 5-A: K-Means cluster purity per position_level

labeled = df[df["position_level"] != "Unknown"].copy()
print(f"Labeled jobs: {len(labeled):,} ({len(labeled)/len(df)*100:.1f}%)")
print()
print(labeled["position_level"].value_counts().to_string())
print()

print("=== K-Means Cluster Purity per Position Level ===")
print(f"{'Position Level':<24}  {'n':>6}  {'Dom. Cluster':>12}  {'Purity':>7}")
print("-" * 58)

purity_rows = []
for pl in labeled["position_level"].value_counts().index:
    sub = labeled[labeled["position_level"] == pl]
    vc  = sub["kmeans_cluster"].value_counts()
    dom_cluster = vc.index[0]
    purity = vc.iloc[0] / len(sub)
    print(f"{pl:<24}  {len(sub):>6,}  {dom_cluster:>12}  {purity:>7.3f}")
    purity_rows.append({"position_level": pl, "n": len(sub), "purity": purity})

mean_purity = np.mean([r["purity"] for r in purity_rows])
random_baseline = 1 / best_k
print()
print(f"Mean purity: {mean_purity:.3f}")
print(f"Random baseline (1/k={best_k}): {random_baseline:.3f}")
print(f"v1 baseline (BGE-small, k=20): 0.164")
print(f"Improvement vs v1: {mean_purity - 0.164:+.3f}")

In [ ]:
# Cell 5-A2: min_years_experience spread by position_level

level_order = (
    labeled.groupby("position_level")["min_years_experience"]
    .median()
    .sort_values()
    .index.tolist()
)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle(f"Years Experience by Position Level — {MODEL_NAME}", fontsize=12)

# Box plot
ax = axes[0]
data_by_level = [
    labeled.loc[labeled["position_level"] == lvl, "min_years_experience"].dropna().values
    for lvl in level_order
]
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(level_order)))
bp = ax.boxplot(data_by_level, vert=True, patch_artist=True, showfliers=False)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
ax.set_xticks(range(1, len(level_order)+1))
ax.set_xticklabels(level_order, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("min_years_experience")
ax.set_title("Box plot")

# Violin plot
ax = axes[1]
data_ok = [(lvl, d) for lvl, d in zip(level_order, data_by_level) if len(d) >= 3]
vp = ax.violinplot([d for _, d in data_ok], showmedians=True)
for pc, color in zip(vp["bodies"], colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
ax.set_xticks(range(1, len(data_ok)+1))
ax.set_xticklabels([lvl for lvl, _ in data_ok], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("min_years_experience")
ax.set_title("Violin plot")

# Median bar
ax = axes[2]
medians = [np.median(d) if len(d) else 0 for d in data_by_level]
bars = ax.barh(level_order, medians, color=colors)
ax.set_xlabel("Median min_years_experience")
ax.set_title("Median by level")

plt.tight_layout()
out = OUTPUT_DIR / "experience_by_level.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

In [ ]:
# Cell 5-B: UMAP scatter of labeled-only points with convex hulls

fig, ax = plt.subplots(figsize=(12, 9))

ax.scatter(
    df.loc[df["position_level"] == "Unknown", "umap_x"],
    df.loc[df["position_level"] == "Unknown", "umap_y"],
    c="#EEEEEE", s=1, alpha=0.1, linewidths=0, rasterized=True
)

for pl, color in PL_PALETTE.items():
    if pl == "Unknown":
        continue
    sub = labeled[labeled["position_level"] == pl]
    if len(sub) < 3:
        continue
    pts = sub[["umap_x", "umap_y"]].values
    ax.scatter(pts[:, 0], pts[:, 1], c=color, s=4, alpha=0.6,
               linewidths=0, label=f"{pl} (n={len(sub):,})", rasterized=True)
    try:
        hull = ConvexHull(pts)
        for simplex in hull.simplices:
            ax.plot(pts[simplex, 0], pts[simplex, 1], color=color, alpha=0.3, lw=0.5)
    except Exception:
        pass

ax.legend(fontsize=8, loc="lower right", markerscale=2)
ax.set_title("UMAP — labeled jobs only, with convex hulls", fontsize=12)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
out = OUTPUT_DIR / "umap_labeled_hulls.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

## Section 6 — KNN Label Propagation

Collapse 9 MCF position levels into 4 broad tiers, train 5-NN cosine classifier,
propagate to all jobs.

**Tier map:**
- T1_Entry: Fresh/entry level, Non-executive
- T2_Junior: Junior Executive, Executive
- T3_Senior: Professional, Senior Executive
- T4_Management: Manager, Middle Management, Senior Management, C-Suite/VP

In [ ]:
# Cell 6-A: Build labeled dataset with tier labels

TIER_MAP = {
    "Fresh/entry level":  "T1_Entry",
    "Non-executive":      "T1_Entry",
    "Junior Executive":   "T2_Junior",
    "Executive":          "T2_Junior",
    "Professional":       "T3_Senior",
    "Senior Executive":   "T3_Senior",
    "Manager":            "T4_Management",
    "Middle Management":  "T4_Management",
    "Senior Management":  "T4_Management",
    "C-Suite/VP":         "T4_Management",
}

labeled_tiered = labeled.copy()
labeled_tiered["tier"] = labeled_tiered["position_level"].map(TIER_MAP)
labeled_tiered = labeled_tiered[labeled_tiered["tier"].notna()].copy()

labeled_idx = labeled_tiered.index.tolist()
X_labeled = X[labeled_idx]
y_labeled = labeled_tiered["tier"].values

print(f"Labeled training samples: {len(X_labeled):,}")
print(pd.Series(y_labeled).value_counts().to_string())

In [ ]:
# Cell 6-B: Train KNN + 5-fold cross-validation
# n_jobs=1 required on Windows with cosine metric to avoid joblib crash

knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="cosine",
    algorithm="brute",
    n_jobs=1,
)

print("Running 5-fold cross-validation (balanced_accuracy)...")
cv_scores = cross_val_score(
    knn, X_labeled, y_labeled, cv=5,
    scoring="balanced_accuracy", n_jobs=1
)
print(f"CV balanced_accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"Per fold: {cv_scores}")
print()
print("v1 baseline (BGE-small): ~0.35 balanced_accuracy")
print(f"Improvement vs v1: {cv_scores.mean() - 0.35:+.3f}")

In [ ]:
# Cell 6-C: Predict tiers for all jobs

print("Fitting KNN on all labeled examples...")
knn.fit(X_labeled, y_labeled)

print("Predicting tiers (may take 2-5 min for large datasets at 768-dim)...")
df["predicted_tier"]       = knn.predict(X)
df["predicted_tier_proba"] = knn.predict_proba(X).max(axis=1)

print("\n=== Predicted Tier Distribution ===")
print(df["predicted_tier"].value_counts().to_string())
print(f"\nMean prediction confidence: {df['predicted_tier_proba'].mean():.3f}")
print(f"Median prediction confidence: {df['predicted_tier_proba'].median():.3f}")

In [ ]:
# Cell 6-D: Visualise propagation results

TIER_COLORS = {
    "T1_Entry":      "#2196F3",
    "T2_Junior":     "#4CAF50",
    "T3_Senior":     "#FF9800",
    "T4_Management": "#F44336",
}

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle(f"KNN Tier Propagation — {MODEL_NAME} ({EMBEDDING_DIMS}-dim)", fontsize=13)

# Plot 1: UMAP coloured by predicted tier
ax = axes[0]
for tier, color in TIER_COLORS.items():
    mask = df["predicted_tier"] == tier
    ax.scatter(df.loc[mask, "umap_x"], df.loc[mask, "umap_y"],
               c=color, s=1, alpha=0.3, linewidths=0,
               label=f"{tier} (n={mask.sum():,})", rasterized=True)
ax.legend(fontsize=8, loc="lower right", markerscale=4)
ax.set_title("Predicted Tier (all jobs)", fontsize=11)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")

# Plot 2: Prediction confidence
ax = axes[1]
sc = ax.scatter(df["umap_x"], df["umap_y"],
                c=df["predicted_tier_proba"], cmap="RdYlGn",
                s=1, alpha=0.3, vmin=0.3, vmax=1.0, linewidths=0, rasterized=True)
plt.colorbar(sc, ax=ax, label="Max class probability")
ax.set_title("Prediction Confidence", fontsize=11)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")

# Plot 3: Salary by predicted tier (validation)
ax = axes[2]
tier_order = ["T1_Entry", "T2_Junior", "T3_Senior", "T4_Management"]
sal_data = [
    df.loc[(df["predicted_tier"] == t) & df["salary_min"].notna(), "salary_min"].values
    for t in tier_order
]
bp = ax.boxplot(sal_data, labels=tier_order, patch_artist=True, showfliers=False)
for patch, color in zip(bp["boxes"], TIER_COLORS.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("salary_min (SGD/month)")
ax.set_title("Salary distribution by predicted tier", fontsize=11)

plt.tight_layout()
out = OUTPUT_DIR / "knn_tier_propagation.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

## Section 7 — Save Artifacts

In [ ]:
# Cell 7: Save all artifacts

# K-Means model
km_path = OUTPUT_DIR / "kmeans_model_v2.pkl"
with open(km_path, "wb") as f:
    pickle.dump(km, f)
print(f"Saved K-Means (k={best_k}) → {km_path}")

# KNN tier classifier
knn_path = OUTPUT_DIR / "knn_tier_classifier_v2.pkl"
with open(knn_path, "wb") as f:
    pickle.dump(knn, f)
print(f"Saved KNN → {knn_path}")

# Enrich cluster profiles with tier distribution
for profile in cluster_profiles:
    c = profile["cluster"]
    sub = df[df["kmeans_cluster"] == c]
    profile["predicted_tier_distribution"] = sub["predicted_tier"].value_counts().to_dict()
    profile["dominant_predicted_tier"] = sub["predicted_tier"].value_counts().idxmax()

profiles_path = OUTPUT_DIR / "cluster_profiles_v2.json"
with open(profiles_path, "w") as f:
    json.dump(cluster_profiles, f, indent=2)
print(f"Saved cluster profiles → {profiles_path}")

# Summary stats
summary = {
    "model": MODEL_NAME,
    "dims": EMBEDDING_DIMS,
    "total_jobs": len(df),
    "labeled_jobs": int((df["position_level"] != "Unknown").sum()),
    "best_k": best_k,
    "best_silhouette": round(silhouette_scores[best_k], 4),
    "mean_cluster_purity": round(mean_purity, 4),
    "knn_cv_balanced_accuracy_mean": round(float(cv_scores.mean()), 4),
    "knn_cv_balanced_accuracy_std": round(float(cv_scores.std()), 4),
    "v1_baseline_mean_purity": 0.164,
    "v1_baseline_knn_cv": 0.35,
}

summary_path = OUTPUT_DIR / "summary_v2.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved summary → {summary_path}")
print()
print(json.dumps(summary, indent=2))

## Section 8 — Conclusions

Fill in after running the notebook.

---

### Comparison vs v1 (BGE-small, 384-dim, no seniority text)

| Metric | v1 (BGE-small) | v2 (BGE-base) | Change |
|--------|---------------|--------------|--------|
| Embedding dims | 384 | 768 | +2x |
| Seniority in text | No | Yes | ✓ |
| Best k | 20 | TBD | — |
| Best silhouette | ~0.054 | TBD | — |
| Mean cluster purity | 0.164 | TBD | — |
| KNN CV balanced_acc | ~0.35 | TBD | — |

---

### Q1: Does the embedding space naturally separate by experience level?
*(answer from umap_overview.png)*

### Q2: Do labeled jobs cluster together by position_level?
*(answer from cluster purity table — mean purity vs 0.164 baseline)*

### Q3: What k produces the best silhouette?
*(answer from silhouette_sweep.png)*

### Q4: Can a KNN classifier propagate tiers with better accuracy than v1?
*(answer from CV balanced_accuracy vs 0.35 baseline)*